# 🗺️ Heatmap de Inversiones Públicas en el Perú (2026)

Visualización geoespacial de los proyectos de inversión pública activos según el banco de proyectos INVIERTE.PE.

**Fuente de datos:** MEF – Sistema de Seguimiento de Inversiones (SSI)  
**Autor:** Leo Molina (@leydimolina20)

---
▶️ **Ejecuta solo las 2 celdas de abajo en orden y listo.**

In [ ]:
# ── CELDA 1: Instalar dependencias (solo la primera vez) ─────────────────────
!pip install folium --quiet

In [ ]:
# ── CELDA 2: Todo el pipeline en un solo bloque ──────────────────────────────
import requests, io, gzip, warnings
import pandas as pd
import folium
from folium.plugins import HeatMap, Fullscreen, MiniMap
warnings.filterwarnings('ignore')

# ── 1. Carga del dataset desde GitHub ────────────────────────────────────────
print('⏳ Descargando datos...')
BASE = 'https://raw.githubusercontent.com/leydimolina20/peru-investment-headmap/main/data'
parts = [
    f'{BASE}/DETALLE_INVERSIONES_part1of3.gz',
    f'{BASE}/DETALLE_INVERSIONES_part2of3.gz',
    f'{BASE}/DETALLE_INVERSIONES_part3of3.gz',
]
raw = b''.join(requests.get(url).content for url in parts)
df = pd.read_csv(
    io.BytesIO(gzip.decompress(raw)),
    encoding='latin1',
    dtype={'LATITUD': float, 'LONGITUD': float, 'MONTO_VIABLE': float},
    low_memory=False
)
print(f'✅ Datos cargados: {len(df):,} proyectos, {df.shape[1]} columnas')

# ── 2. Limpieza y filtrado ────────────────────────────────────────────────────
df_geo = df.dropna(subset=['LATITUD', 'LONGITUD']).copy()
df_geo = df_geo[
    df_geo['LATITUD'].between(-20, 0) &
    df_geo['LONGITUD'].between(-82, -68)
]
print(f'📍 Proyectos con coordenadas válidas: {len(df_geo):,}')

# ── 3. Peso por monto viable ──────────────────────────────────────────────────
monto_max = df_geo['MONTO_VIABLE'].quantile(0.95)
df_geo['peso'] = (df_geo['MONTO_VIABLE'].clip(upper=monto_max) / monto_max).fillna(0.1)
heat_data = df_geo[['LATITUD', 'LONGITUD', 'peso']].values.tolist()

# ── 4. Mapa base ──────────────────────────────────────────────────────────────
mean_lat = df_geo['LATITUD'].mean()
mean_lon = df_geo['LONGITUD'].mean()

m = folium.Map(location=[mean_lat, mean_lon], zoom_start=5, tiles='CartoDB dark_matter')

# ── 5. Capa de calor ──────────────────────────────────────────────────────────
HeatMap(
    data=heat_data,
    radius=8,
    blur=10,
    max_zoom=13,
    max_val=1.0,
    gradient={0.2: '#1565C0', 0.4: '#26A69A', 0.6: '#FFC107', 0.85: '#FF5722', 1.0: '#B71C1C'}
).add_to(m)

# ── 6. Controles ──────────────────────────────────────────────────────────────
Fullscreen(position='topright', title='Pantalla completa',
           title_cancel='Salir', force_separate_button=True).add_to(m)
MiniMap(toggle_display=True, position='bottomleft').add_to(m)

# ── 7. Título, fuente y leyenda ───────────────────────────────────────────────
m.get_root().html.add_child(folium.Element("""
<div style="position:fixed;top:10px;left:50px;z-index:9999;
    background:rgba(0,0,0,0.78);padding:12px 18px;border-radius:8px;
    color:white;font-family:Arial;box-shadow:0 2px 8px rgba(0,0,0,0.4)">
  <h3 style="margin:0;font-size:18px">🏗️ Densidad de Inversión Pública en el Perú</h3>
  <p style="margin:4px 0 0 0;font-size:12px;color:#90CAF9">
    Proyectos activos INVIERTE.PE · Año 2026 · Intensidad = Monto viable</p>
</div>"""))

m.get_root().html.add_child(folium.Element("""
<div style="position:fixed;top:10px;right:50px;z-index:9999;
    color:#ccc;font-family:Arial;font-size:11px;
    background:rgba(0,0,0,0.55);padding:6px 10px;border-radius:5px">
  Fuente: MEF – SSI | Elaborado por: Leo Molina
</div>"""))

m.get_root().html.add_child(folium.Element("""
<div style="position:fixed;bottom:60px;right:10px;z-index:9999;
    background:rgba(0,0,0,0.75);padding:10px 14px;border-radius:7px;
    color:white;font-family:Arial;font-size:11px">
  <b>Intensidad de inversión</b><br>
  <span style='color:#B71C1C'>■</span> Muy alta<br>
  <span style='color:#FF5722'>■</span> Alta<br>
  <span style='color:#FFC107'>■</span> Media<br>
  <span style='color:#26A69A'>■</span> Baja<br>
  <span style='color:#1565C0'>■</span> Muy baja
</div>"""))

# ── 8. Guardar y mostrar ──────────────────────────────────────────────────────
m.save('peru_investment_heatmap.html')
print('✅ Mapa guardado como: peru_investment_heatmap.html')
m

In [ ]:
# ── CELDA 3 (opcional): Resumen por departamento ─────────────────────────────
resumen = (
    df_geo
    .groupby('DEPARTAMENTO')
    .agg(proyectos=('CODIGO_UNICO','count'),
         monto_total=('MONTO_VIABLE','sum'),
         monto_promedio=('MONTO_VIABLE','mean'))
    .sort_values('monto_total', ascending=False)
    .reset_index()
)
resumen['monto_total_MM']   = (resumen['monto_total']   / 1e6).round(2)
resumen['monto_promedio_MM']= (resumen['monto_promedio'] / 1e6).round(3)
print('Top 15 departamentos por monto viable total (millones S/.)')
resumen[['DEPARTAMENTO','proyectos','monto_total_MM','monto_promedio_MM']].head(15)